In [0]:
df = spark.read.json("/Volumes/linkedin_scraper_etl/live_linkedin/linkedin_raw/json/linkedin_jobs_latest.json")
display(df)

In [0]:
json_file = "/Workspace/Shared/linkedin_scraper/CareerFlowEngine/src/data/patterns_deduped.json"
patterns_df = spark.read.json(json_file)
display(patterns_df)

In [0]:
import spacy
import json
import pandas as pd
import re

# Load spacy model
nlp = spacy.load("en_core_web_trf")

# Add your EntityRuler with custom patterns
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Convert PySpark DataFrame to pandas DataFrame
patterns_pd = patterns_df.toPandas()

# If your patterns are in a DataFrame called patterns_df
json_str = patterns_pd.to_json(
    orient='records'
)

# Convert JSON string to Python list of dicts
patterns_list = json.loads(json_str)

ruler.add_patterns(patterns_list)  # from your JSON

# ======================
# Clean text helper
# ======================
# def clean_text(text):
#     if not text:
#         return ""
#     # Normalize unicode dashes/arrows to plain ASCII
#     text = re.sub(r'[→⇒➝➔➡➞➟➠➤➧➩➮➯➱➲➳➵➸➺➻➼➽➾]', '->', text)

#     # Replace variations of "->" with space
#     text = re.sub(r'\s*-\s*>\s*', ' ', text)

#     # text = re.sub(r'[^a-zA-Z0-9\s\+\.\-\/#]', ' ', text)

#     # Collapse multiple spaces
#     text = re.sub(r'\s+', ' ', text)

#     return text.strip()

def clean_text(text):
    if not text:
        return ""
    # Remove unwanted special characters (add more if needed)
    patterns_to_remove = [r'->']
    for pat in patterns_to_remove:
        if re.search(pat, text):
            text = re.sub(pat, ' ', text)
    return text

# Function: extract skills & locations separately
def extract_entities(text):
    cleaned = clean_text(text)

    # Debug: show what goes into spaCy
    # print("\n============================")
    # print("RAW TEXT:\n", text[:200], "...")
    # print("CLEANED TEXT:\n", cleaned[:200],"...")
          
    doc = nlp(cleaned if cleaned else "")
    
    # Debug: show how spaCy tokenizes
    # print("TOKENS:", [t.text for t in doc])


    skills = [ent.text for ent in doc.ents if ent.label_ in ["CLOUD_VENDOR","CLOUD_TECH","API Skills","Skill"]]
    locations = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
    skills = list(dict.fromkeys(skills))
    locations = list(dict.fromkeys(locations))

    # print("SKILLS:", skills)
    # print("LOCATIONS:", locations)

    return skills, locations

In [0]:
job_pd = df.toPandas()

job_pd["skills"], job_pd["locations"] = zip(*job_pd["raw_description"].apply(extract_entities))

In [0]:
job_df = spark.createDataFrame(job_pd)
job_df.write.mode("append").saveAsTable("career_flow_engine.bronze.careerflow_raw")
display(job_df)

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr

# Define retention period
retention_days = 60
target_table = "career_flow_engine.bronze.careerflow_raw"
# Filter out records older than retention period
delete_sql = f"""
DELETE FROM {target_table}
WHERE scraped_at < DATEADD(day, -{retention_days}, CURRENT_DATE())
"""

spark.sql(delete_sql)